# Tuning de Hiperparâmetros — XGBoost

Notebook para explorar combinações de hiperparâmetros do XGBClassifier.

**Fluxo:**
1. Carrega dados processados
2. Divide em treino / validação / teste (estratificado)
3. Testa combinações via `ParameterGrid` na validação (F1 Score)
4. Retreina com melhores params no treino completo
5. Avalia no teste holdout

Ajuste o `PARAM_GRID` e reexecute as cells a partir da cell 4 a cada iteração.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    matthews_corrcoef,
)
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier

from config import (
    PROCESSED_TRAIN_FILE,
    TARGET_COL,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
)

print("Imports OK")

Imports OK


## 1. Carregar dados e dividir em treino / validação / teste

In [2]:
# =============================================
# HIPERPARÂMETROS DE SPLIT — ajuste se quiser
# =============================================
RANDOM_STATE = 42
TEST_SIZE = 0.20
VALIDATION_SIZE = 0.10

# Carregar dados processados
df = pd.read_csv(PROCESSED_TRAIN_FILE)
print(f"Dataset: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"Distribuição do target:")
print(df[TARGET_COL].value_counts().rename({1: "Admissão (1)", -1: "Desligamento (-1)"}))

# Separar features e target
feature_cols = list(NUMERIC_FEATURES) + list(CATEGORICAL_FEATURES)
X = df[feature_cols].copy()
y = df[TARGET_COL].map({1: 1, -1: 0})  # XGBoost espera labels >= 0

# Split 1: treino_completo (80%) + teste (20%)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# Split 2: treino (90% do treino_completo) + validação (10%)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=VALIDATION_SIZE, random_state=RANDOM_STATE, stratify=y_train_full
)

print(f"\nTamanhos:")
print(f"  Treino       : {X_train.shape[0]:>10,}")
print(f"  Validação    : {X_val.shape[0]:>10,}")
print(f"  Treino compl.: {X_train_full.shape[0]:>10,}")
print(f"  Teste        : {X_test.shape[0]:>10,}")

Dataset: 738,837 linhas × 21 colunas
Distribuição do target:
saldomovimentacao
Desligamento (-1)    377254
Admissão (1)         361583
Name: count, dtype: int64

Tamanhos:
  Treino       :    531,962
  Validação    :     59,107
  Treino compl.:    591,069
  Teste        :    147,768


In [3]:
resumo = df.nunique().to_frame("n_unicos")
resumo["tipo"] = df.dtypes

display(resumo.sort_values(by="n_unicos", ascending=False))

,n_unicos,tipo
salario,60305,float64
cbo2002ocupacao,1841,int64
horascontratuais,1099,float64
subclasse,1035,int64
idade,76,float64
secao,21,str
municipio,16,int64
graudeinstrucao,13,int64
tamestabjan,10,int64
racacor,7,int64


## 2. Preprocessador

In [4]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="desconhecido")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, list(NUMERIC_FEATURES)),
        ("cat", categorical_pipeline, list(CATEGORICAL_FEATURES)),
    ],
    remainder="drop",
)

print("Preprocessador montado.")

Preprocessador montado.


## 3. Função de Grid Search

Função reutilizável que recebe um `param_grid`, executa o grid search na validação e exibe o ranking.
Chame quantas vezes quiser com parâmetros diferentes — cada execução fica registrada como justificativa.

In [5]:
def run_grid_search(param_grid: dict) -> tuple[dict, float, pd.DataFrame]:
    """
    Executa grid search manual na validação e exibe o ranking dos resultados.

    Parameters:
        param_grid: Dicionário com hiperparâmetros a testar (formato ParameterGrid).

    Returns:
        Tupla (melhores_params, melhor_f1, dataframe_ranking).
    """
    grid = list(ParameterGrid(param_grid))
    total = len(grid)
    print(f"Total de combinações a testar: {total}")
    print("=" * 80)

    results = []
    best_score = -np.inf
    best_params = None

    for i, params in enumerate(grid, 1):
        clf = XGBClassifier(
            **params,
            objective="binary:logistic",
            eval_metric="logloss",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )

        pipe = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("classifier", clf),
        ])

        pipe.fit(X_train, y_train)
        y_val_pred = pipe.predict(X_val)
        score = f1_score(y_val, y_val_pred)

        marker = " *** NOVO MELHOR" if score > best_score else ""
        print(f"[{i:>{len(str(total))}}/{total}] F1: {score:.4f}{marker} | {params}")

        results.append({"f1_val": score, **params})

        if score > best_score:
            best_score = score
            best_params = params

    print("=" * 80)
    print(f"\nMelhores hiperparâmetros: {best_params}")
    print(f"Melhor F1 (validação): {best_score:.4f}")

    # Ranking
    df_results = pd.DataFrame(results).sort_values("f1_val", ascending=False).reset_index(drop=True)
    df_results.index += 1
    df_results.index.name = "rank"
    display(df_results)

    return best_params, best_score, df_results

print("Função run_grid_search() definida.")

Função run_grid_search() definida.


## 4. Execuções do Grid Search

Cada cell abaixo é uma execução com parâmetros diferentes.
Duplique a cell, ajuste o grid e execute novamente — o histórico fica registrado.

### Execução 1 — Exploração inicial ampla

Grade ampla para identificar regiões promissoras do espaço de hiperparâmetros. Sem conclusões prévias para guiar os parâmetros escolhidos.

In [ ]:
best_params, best_score, df_results = run_grid_search({
    "n_estimators": [300, 500],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.05, 0.1],
    "min_child_weight": [1, 5],
    "subsample": [0.8],
    "colsample_bytree": [0.8],
})

Total de combinações a testar: 24
[ 1/24] F1: 0.5883 *** NOVO MELHOR | {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 4, 'min_child_weight': 1, 'n_estimators': 300, 'subsample': 0.8}
[ 2/24] F1: 0.5921 *** NOVO MELHOR | {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 4, 'min_child_weight': 1, 'n_estimators': 500, 'subsample': 0.8}
[ 3/24] F1: 0.5894 | {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 4, 'min_child_weight': 5, 'n_estimators': 300, 'subsample': 0.8}
[ 4/24] F1: 0.5913 | {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 4, 'min_child_weight': 5, 'n_estimators': 500, 'subsample': 0.8}
[ 5/24] F1: 0.6022 *** NOVO MELHOR | {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 6, 'min_child_weight': 1, 'n_estimators': 300, 'subsample': 0.8}
[ 6/24] F1: 0.6088 *** NOVO MELHOR | {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 6, 'min_child_weight': 1, 'n_estimators': 500, 'subsample': 0.8}
[ 7/24] F1: 0.

,f1_val,colsample_bytree,learning_rate,max_depth,min_child_weight,n_estimators,subsample
rank,,,,,,,
1,0.620868,0.8,0.10,8,5,500,0.8
2,0.620411,0.8,0.10,8,1,500,0.8
3,0.620229,0.8,0.05,8,1,500,0.8
4,0.619653,0.8,0.05,8,5,500,0.8
5,0.619589,0.8,0.10,8,1,300,0.8
6,0.618264,0.8,0.10,8,5,300,0.8
7,0.614316,0.8,0.10,6,1,500,0.8
8,0.613233,0.8,0.10,6,5,500,0.8
9,0.613087,0.8,0.05,8,1,300,0.8


### Execução 2 — Expansão em profundidade e estimadores

A execução 1 indicou que `max_depth` mais profundo e `n_estimators` maiores consistentemente melhoravam o F1, com ambos na borda superior da grade — sinal de que havia espaço para explorar além. Expandiu-se também `subsample` e `colsample_bytree`. Os resultados mostraram que `max_depth=12` com `learning_rate=0.1` causa overfitting claro (F1 cai com mais estimadores), enquanto `learning_rate=0.05` se mostrou mais estável em profundidades maiores. `colsample_bytree=0.7` apresentou leve vantagem sobre `0.8`.

In [ ]:
best_params, best_score, df_results = run_grid_search({
    "n_estimators": [500, 700, 1000],
    "max_depth": [8, 10, 12],
    "learning_rate": [0.05, 0.1],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.7, 0.8],
    "colsample_bytree": [0.7, 0.8],
})

Total de combinações a testar: 216
[  1/216] F1: 0.6185 *** NOVO MELHOR | {'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 8, 'min_child_weight': 1, 'n_estimators': 500, 'subsample': 0.7}
[  2/216] F1: 0.6189 *** NOVO MELHOR | {'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 8, 'min_child_weight': 1, 'n_estimators': 500, 'subsample': 0.8}
[  3/216] F1: 0.6200 *** NOVO MELHOR | {'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 8, 'min_child_weight': 1, 'n_estimators': 700, 'subsample': 0.7}
[  4/216] F1: 0.6215 *** NOVO MELHOR | {'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 8, 'min_child_weight': 1, 'n_estimators': 700, 'subsample': 0.8}
[  5/216] F1: 0.6211 | {'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 8, 'min_child_weight': 1, 'n_estimators': 1000, 'subsample': 0.7}
[  6/216] F1: 0.6243 *** NOVO MELHOR | {'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 8, 'min_child_weight': 1, 'n_estimators': 1000, '

,f1_val,colsample_bytree,learning_rate,max_depth,min_child_weight,n_estimators,subsample
rank,,,,,,,
1,0.626329,0.7,0.05,12,5,500,0.8
2,0.625824,0.7,0.05,10,1,500,0.7
3,0.625625,0.7,0.05,10,3,500,0.8
4,0.625574,0.7,0.05,10,3,700,0.8
5,0.625215,0.8,0.05,10,5,700,0.7
...,...,...,...,...,...,...,...
212,0.604342,0.8,0.10,12,3,1000,0.7
213,0.603215,0.7,0.10,12,1,1000,0.7
214,0.602577,0.7,0.10,12,1,1000,0.8


### Execução 3 — Refinamento: learning rate menor e colsample reduzido

A execução 2 sugeriu que menor `colsample_bytree` e menor `learning_rate` favoreciam a generalização em profundidades maiores. Introduziu-se `learning_rate=0.03` e `colsample_bytree=0.6`, que dominaram o topo do ranking, confirmando a hipótese.

In [ ]:
best_params, best_score, df_results = run_grid_search({
    "n_estimators": [400, 500, 600],
    "max_depth": [10, 11, 12],
    "learning_rate": [0.03, 0.05],
    "min_child_weight": [5, 7, 10],
    "subsample": [0.8, 0.9],
    "colsample_bytree": [0.6, 0.7],
})

Total de combinações a testar: 216
[  1/216] F1: 0.6217 *** NOVO MELHOR | {'colsample_bytree': 0.6, 'learning_rate': 0.03, 'max_depth': 10, 'min_child_weight': 5, 'n_estimators': 400, 'subsample': 0.8}
[  2/216] F1: 0.6216 | {'colsample_bytree': 0.6, 'learning_rate': 0.03, 'max_depth': 10, 'min_child_weight': 5, 'n_estimators': 400, 'subsample': 0.9}
[  3/216] F1: 0.6230 *** NOVO MELHOR | {'colsample_bytree': 0.6, 'learning_rate': 0.03, 'max_depth': 10, 'min_child_weight': 5, 'n_estimators': 500, 'subsample': 0.8}
[  4/216] F1: 0.6236 *** NOVO MELHOR | {'colsample_bytree': 0.6, 'learning_rate': 0.03, 'max_depth': 10, 'min_child_weight': 5, 'n_estimators': 500, 'subsample': 0.9}
[  5/216] F1: 0.6240 *** NOVO MELHOR | {'colsample_bytree': 0.6, 'learning_rate': 0.03, 'max_depth': 10, 'min_child_weight': 5, 'n_estimators': 600, 'subsample': 0.8}
[  6/216] F1: 0.6250 *** NOVO MELHOR | {'colsample_bytree': 0.6, 'learning_rate': 0.03, 'max_depth': 10, 'min_child_weight': 5, 'n_estimators': 60

,f1_val,colsample_bytree,learning_rate,max_depth,min_child_weight,n_estimators,subsample
rank,,,,,,,
1,0.628453,0.6,0.03,12,5,600,0.8
2,0.628141,0.7,0.03,12,5,600,0.9
3,0.627793,0.7,0.03,11,5,600,0.9
4,0.627731,0.6,0.03,12,10,600,0.9
5,0.627715,0.7,0.03,12,10,600,0.9
...,...,...,...,...,...,...,...
212,0.620298,0.6,0.03,10,10,400,0.9
213,0.619749,0.7,0.03,10,5,400,0.8
214,0.619192,0.7,0.03,10,7,400,0.8


### Execução 4 — Refinamento fino ao redor do melhor

A execução 3 consolidou `colsample_bytree=0.6, learning_rate=0.03, max_depth=12, n_estimators=600` como região ótima. Exploração estreita nessa vizinhança revelou que `max_depth=13` com `subsample=0.9` superou `max_depth=12`, sugerindo ganho residual com árvores ligeiramente mais profundas.

In [ ]:
best_params, best_score, df_results = run_grid_search({
    "n_estimators": [600, 700, 800],
    "max_depth": [12, 13],
    "learning_rate": [0.03],
    "min_child_weight": [5, 7],
    "subsample": [0.8, 0.9],
    "colsample_bytree": [0.6],
})

Total de combinações a testar: 24
[ 1/24] F1: 0.6285 *** NOVO MELHOR | {'colsample_bytree': 0.6, 'learning_rate': 0.03, 'max_depth': 12, 'min_child_weight': 5, 'n_estimators': 600, 'subsample': 0.8}
[ 2/24] F1: 0.6268 | {'colsample_bytree': 0.6, 'learning_rate': 0.03, 'max_depth': 12, 'min_child_weight': 5, 'n_estimators': 600, 'subsample': 0.9}
[ 3/24] F1: 0.6266 | {'colsample_bytree': 0.6, 'learning_rate': 0.03, 'max_depth': 12, 'min_child_weight': 5, 'n_estimators': 700, 'subsample': 0.8}
[ 4/24] F1: 0.6266 | {'colsample_bytree': 0.6, 'learning_rate': 0.03, 'max_depth': 12, 'min_child_weight': 5, 'n_estimators': 700, 'subsample': 0.9}
[ 5/24] F1: 0.6254 | {'colsample_bytree': 0.6, 'learning_rate': 0.03, 'max_depth': 12, 'min_child_weight': 5, 'n_estimators': 800, 'subsample': 0.8}
[ 6/24] F1: 0.6267 | {'colsample_bytree': 0.6, 'learning_rate': 0.03, 'max_depth': 12, 'min_child_weight': 5, 'n_estimators': 800, 'subsample': 0.9}
[ 7/24] F1: 0.6265 | {'colsample_bytree': 0.6, 'learning

,f1_val,colsample_bytree,learning_rate,max_depth,min_child_weight,n_estimators,subsample
rank,,,,,,,
1,0.629023,0.6,0.03,13,5,800,0.9
2,0.628477,0.6,0.03,13,5,700,0.9
3,0.628453,0.6,0.03,12,5,600,0.8
4,0.628268,0.6,0.03,13,5,600,0.9
5,0.628075,0.6,0.03,13,7,600,0.9
6,0.627508,0.6,0.03,12,7,600,0.9
7,0.627412,0.6,0.03,12,7,800,0.9
8,0.627193,0.6,0.03,12,7,700,0.9
9,0.626777,0.6,0.03,12,5,600,0.9



### Execução 5 — Verificação do teto

A execução 4 confirmou `max_depth=13` como promissor, ainda na borda superior da grade. Testou-se `max_depth=14` e `n_estimators` maiores para verificar se havia ganho adicional. Ambos pioraram o F1, confirmando overfitting e indicando que os parâmetros da execução 4 representam o teto do modelo.

In [6]:
best_params, best_score, df_results = run_grid_search({
    "n_estimators": [800, 1000, 1200],
    "max_depth": [13, 14],
    "learning_rate": [0.03],
    "min_child_weight": [5],
    "subsample": [0.9],
    "colsample_bytree": [0.6],
})

Total de combinações a testar: 6
[1/6] F1: 0.6290 *** NOVO MELHOR | {'colsample_bytree': 0.6, 'learning_rate': 0.03, 'max_depth': 13, 'min_child_weight': 5, 'n_estimators': 800, 'subsample': 0.9}
[2/6] F1: 0.6275 | {'colsample_bytree': 0.6, 'learning_rate': 0.03, 'max_depth': 13, 'min_child_weight': 5, 'n_estimators': 1000, 'subsample': 0.9}
[3/6] F1: 0.6264 | {'colsample_bytree': 0.6, 'learning_rate': 0.03, 'max_depth': 13, 'min_child_weight': 5, 'n_estimators': 1200, 'subsample': 0.9}
[4/6] F1: 0.6251 | {'colsample_bytree': 0.6, 'learning_rate': 0.03, 'max_depth': 14, 'min_child_weight': 5, 'n_estimators': 800, 'subsample': 0.9}
[5/6] F1: 0.6245 | {'colsample_bytree': 0.6, 'learning_rate': 0.03, 'max_depth': 14, 'min_child_weight': 5, 'n_estimators': 1000, 'subsample': 0.9}
[6/6] F1: 0.6222 | {'colsample_bytree': 0.6, 'learning_rate': 0.03, 'max_depth': 14, 'min_child_weight': 5, 'n_estimators': 1200, 'subsample': 0.9}

Melhores hiperparâmetros: {'colsample_bytree': 0.6, 'learning_ra

,f1_val,colsample_bytree,learning_rate,max_depth,min_child_weight,n_estimators,subsample
rank,,,,,,,
1,0.629023,0.6,0.03,13,5,800,0.9
2,0.627478,0.6,0.03,13,5,1000,0.9
3,0.626414,0.6,0.03,13,5,1200,0.9
4,0.625116,0.6,0.03,14,5,800,0.9
5,0.624489,0.6,0.03,14,5,1000,0.9
6,0.622188,0.6,0.03,14,5,1200,0.9


## 5. Treinar modelo final no treino completo e avaliar no teste holdout

Quando encontrar os melhores parâmetros, execute esta cell para treinar no treino completo e avaliar no teste.

In [18]:
# best_params = {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 8, 'min_child_weight': 5, 'n_estimators': 500, 'subsample': 0.8}   # F1 0.6209
# best_params = {'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 12, 'min_child_weight': 5, 'n_estimators': 500, 'subsample': 0.8} # F1 0.6263
# best_params = {'colsample_bytree': 0.6, 'learning_rate': 0.03, 'max_depth': 12, 'min_child_weight': 5, 'n_estimators': 600, 'subsample': 0.8} # F1 0.6285
best_params = {'colsample_bytree': 0.6, 'learning_rate': 0.03, 'max_depth': 13, 'min_child_weight': 5, 'n_estimators': 800, 'subsample': 0.9} # F1 0.6290

In [7]:
# Treina com os melhores hiperparâmetros no treino COMPLETO
print(f"Treinando modelo final com: {best_params}")
print(f"Treino completo: {X_train_full.shape[0]:,} registros")

final_clf = XGBClassifier(
    **best_params,
    objective="binary:logistic",
    eval_metric="logloss",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

final_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", final_clf),
])

final_pipe.fit(X_train_full, y_train_full)
y_test_pred = final_pipe.predict(X_test)

target_names = ["Desligamento (-1)", "Admissão (1)"]

print("\n" + "=" * 80)
print("AVALIAÇÃO NO TESTE HOLDOUT")
print("=" * 80)
print(f"  F1 Score (binary)    : {f1_score(y_test, y_test_pred):.4f}  ← métrica principal")
print(f"  F1 Score (macro)     : {f1_score(y_test, y_test_pred, average='macro'):.4f}")
print(f"  F1 Score (weighted)  : {f1_score(y_test, y_test_pred, average='weighted'):.4f}")
print(f"  Accuracy             : {accuracy_score(y_test, y_test_pred):.4f}")
print(f"  Balanced Accuracy    : {balanced_accuracy_score(y_test, y_test_pred):.4f}")
print(f"  MCC                  : {matthews_corrcoef(y_test, y_test_pred):.4f}")
print("-" * 80)
print("Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=target_names, digits=4))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

Treinando modelo final com: {'colsample_bytree': 0.6, 'learning_rate': 0.03, 'max_depth': 13, 'min_child_weight': 5, 'n_estimators': 800, 'subsample': 0.9}
Treino completo: 591,069 registros

AVALIAÇÃO NO TESTE HOLDOUT
  F1 Score (binary)    : 0.6263  ← métrica principal
  F1 Score (macro)     : 0.6307
  F1 Score (weighted)  : 0.6308
  Accuracy             : 0.6307
  Balanced Accuracy    : 0.6308
  MCC                  : 0.2615
--------------------------------------------------------------------------------
Classification Report:
                   precision    recall  f1-score   support

Desligamento (-1)     0.6410    0.6292    0.6351     75451
     Admissão (1)     0.6204    0.6323    0.6263     72317

         accuracy                         0.6307    147768
        macro avg     0.6307    0.6308    0.6307    147768
     weighted avg     0.6309    0.6307    0.6308    147768

Confusion Matrix:
[[47477 27974]
 [26591 45726]]


## 6. Copie os melhores parâmetros para o `config.py`

Quando estiver satisfeito com os resultados, copie o `best_params` para o `XGBOOST_PARAM_GRID` do `config.py` (com listas de um só elemento) e rode `python main.py`.

In [ ]:
print("Cole no config.py XGBOOST_PARAM_GRID:")
print()
print("XGBOOST_PARAM_GRID = {")
for k, v in best_params.items():
    print(f'    "{k}": [{repr(v)}],')
print("}")